In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path
from xml.dom import minidom

In [ ]:
def string_to_xml_file(xml_string, file_name):
    """
    Converts a string into a well-formatted (indented) XML file, without unnecessary newlines.
    
    Parameters:
    xml_string (str): The XML content as a string.
    file_name (str): The desired filename for the XML file.
    """
    try:
        # Parse the XML string
        root = ET.ElementTree(ET.fromstring(xml_string))
        
        # Convert ElementTree to a string
        rough_string = ET.tostring(root.getroot(), encoding="utf-8")
        
        # Use minidom to pretty-print the XML
        parsed = minidom.parseString(rough_string)
        pretty_xml_as_string = parsed.toprettyxml(indent="  ")
        
        # Remove unnecessary blank lines created by toprettyxml()
        pretty_xml_as_string = "\n".join([line for line in pretty_xml_as_string.splitlines() if line.strip()])
        
        # Write the formatted XML to a file
        with open(file_name, "w", encoding="utf-8") as f:
            f.write(pretty_xml_as_string)
        
        print(f"XML file '{file_name}' created successfully with proper indentation and no extra newlines.")
    except ET.ParseError as e:
        print("Error parsing XML string:", e)

# Source

* Data Source:
    * [Korea-Pass Platform](https://korea-pass.kr/)

* Implemented Input Files
    * `/input/policy/korea-2035/transportation/K_pass_cp.xml`
    * `/input/policy/korea-2035/transportation/K_pass_ep.xml`

# K-Pass

Korea allocated 235.7 billion KRW to promote public transportation usage. This pays back 20% of public transportation fee. For people in age 19-34, the refund rate increases to 30%.

According to [Korean Government](https://www.molit.go.kr/USR/NEWS/m_35045/dtl.jsp?lcmspage=1&id=95085966), 20s' share is 16.9% and 30s' share is 19% among public trans users. We assume users of 30-34 takes 9.5% share. 26.4% gets 30% discount and the others get 20% discount.

In [20]:
effDiscountRate = 0.264 * 0.3 + (1-0.264) * 0.2
effDiscountRate

0.2264

In average, effective refund rate of K-Pass is modeled as 22.64%. Rail in GCAM includes both the conventional rail and subway. In [Korea](https://www.ktdb.go.kr/www/selectTrnsportTreeView.do?key=30&idx=46190&pageUnit=10&pageIndex=1&searchCnd=all&searchTy=1), conventional railway's service output was 15984 + 5223 (counting only electricity based ones, KTX and SRT) mil passkm in 2022 and that of subway was 35,936. So the share of subway is

In [21]:
subwayShare = 35936 / (15984 + 5223 + 35936)
subwayShare

0.6288784278039304

Only subway fee is refunded so, the effective support rate could be calculated as bleow.

In [ ]:
effDiscountRateRail = effDiscountRate * subwayShare
effDiscountRateRail

0.14237807605480984

In the *Enhanced Ambition* scenario we assume 30% of support rate is applied for all passengers. Detailed implementation steps are provided below.

In [23]:
proj_path = Path("/data/project/tae/gcam-core")
xml_path = proj_path / "input" / "gcamdata" / "xml"
db_path = proj_path / "output"

In [24]:
xml_file_path = xml_path / "transportation_UCD_CORE.xml"
tree = ET.parse(xml_file_path)
root = tree.getroot()  # Get the root element of the XML
korea = root.find(".//region[@name='South Korea']")

In [26]:
# Create the new root for the reproduced XML
new_root = ET.Element("scenario")
new_world = ET.SubElement(new_root, "world")
new_korea = ET.SubElement(new_world, "region", {'name': "South Korea"})
for supplysector in korea.findall(".//supplysector"):
    supplysector_nm = supplysector.get('name')

    new_supplysector = ET.Element('supplysector', {'name': supplysector_nm})

    for subsector in supplysector.findall(".//tranSubsector"):
        subsector_nm = subsector.get('name')
        if subsector_nm not in ['Passenger Rail', 'Bus']:
            continue

        new_subsector = ET.Element('tranSubsector', {'name': subsector_nm})
        
        for stub_technology in subsector.findall(".//stub-technology"):
            stub_technology_nm = stub_technology.get("name")
            if stub_technology_nm not in ["Electric", "BEV"]:
                continue

            new_stub_technology = ET.Element('stub-technology', {'name': stub_technology_nm})
            for period in stub_technology.findall(".//period"):
                year = int(period.get('year'))
                if year not in range(2025, 2040, 5):
                    continue
                

                
                new_period = ET.SubElement(new_stub_technology, 'period', {'year': str(year)})
                
                tracking_non_energy_input = period.find(".//tracking-non-energy-input")
                input_cost = tracking_non_energy_input.find(".//input-cost")
                input_cost_val = float(input_cost.text)

                minicam_non_energy_input = ET.SubElement(new_period, 'minicam-non-energy-input', {'name': 'K-Pass'})
                input_cost = ET.SubElement(minicam_non_energy_input, 'input-cost')
                if year == 2025:
                    input_cost_text = -(input_cost_val * effDiscountRateRail * (3/5) if subsector_nm == "Passenger Rail" else input_cost_val * effDiscountRate  * (3/5))
                    input_cost.text = f"{input_cost_text:.6f}"
                else:
                    input_cost_text = -(input_cost_val * effDiscountRateRail if subsector_nm == "Passenger Rail" else input_cost_val * effDiscountRate)
                    input_cost.text = f"{input_cost_text:.6f}"
            
            if new_stub_technology:
                new_subsector.append(new_stub_technology)
        
        if new_subsector:
            new_supplysector.append(new_subsector)
    if new_supplysector:
        new_korea.append(new_supplysector)

In [28]:
outfile_path = proj_path / "input" / "policy" / "ndc" / "transportation" / "K_pass_cp.xml"

# save
xml_string = ET.tostring(new_root, encoding="unicode")
string_to_xml_file(xml_string, outfile_path)

XML file '/data/project/tae/gcam-core/input/policy/ndc/transportation/K_pass_cp.xml' created successfully with proper indentation and no extra newlines.


## Enhanced Policy

In [32]:
effDiscountRate_eh = 0.3

In [33]:
effDiscountRateRail_eh = effDiscountRate_eh * subwayShare
effDiscountRateRail_eh

0.18866352834117914

In [35]:
# Create the new root for the reproduced XML
new_root = ET.Element("scenario")
new_world = ET.SubElement(new_root, "world")
new_korea = ET.SubElement(new_world, "region", {'name': "South Korea"})
for supplysector in korea.findall(".//supplysector"):
    supplysector_nm = supplysector.get('name')

    new_supplysector = ET.Element('supplysector', {'name': supplysector_nm})

    for subsector in supplysector.findall(".//tranSubsector"):
        subsector_nm = subsector.get('name')
        if subsector_nm not in ['Passenger Rail', 'Bus']:
            continue

        new_subsector = ET.Element('tranSubsector', {'name': subsector_nm})
        
        for stub_technology in subsector.findall(".//stub-technology"):
            stub_technology_nm = stub_technology.get("name")
            if stub_technology_nm not in ["Electric", "BEV"]:
                continue

            new_stub_technology = ET.Element('stub-technology', {'name': stub_technology_nm})
            for period in stub_technology.findall(".//period"):
                year = int(period.get('year'))
                if year not in range(2025, 2040, 5):
                    continue
                
                new_period = ET.SubElement(new_stub_technology, 'period', {'year': str(year)})
                
                tracking_non_energy_input = period.find(".//tracking-non-energy-input")
                input_cost = tracking_non_energy_input.find(".//input-cost")
                input_cost_val = float(input_cost.text)

                minicam_non_energy_input = ET.SubElement(new_period, 'minicam-non-energy-input', {'name': 'K-Pass'})
                input_cost = ET.SubElement(minicam_non_energy_input, 'input-cost')
                if year == 2025:
                    input_cost_text = -(input_cost_val * effDiscountRateRail * (3/5) if subsector_nm == "Passenger Rail" else input_cost_val * effDiscountRate  * (3/5))
                    input_cost.text = f"{input_cost_text:.6f}"
                else:
                    input_cost_text = -(input_cost_val * effDiscountRateRail_eh if subsector_nm == "Passenger Rail" else input_cost_val * effDiscountRate_eh)
                    input_cost.text = f"{input_cost_text:.6f}"
            
            if new_stub_technology:
                new_subsector.append(new_stub_technology)
        
        if new_subsector:
            new_supplysector.append(new_subsector)
    if new_supplysector:
        new_korea.append(new_supplysector)

In [36]:
outfile_path = proj_path / "input" / "policy" / "ndc" / "transportation" / "K_pass_ep.xml"

# save
xml_string = ET.tostring(new_root, encoding="unicode")
string_to_xml_file(xml_string, outfile_path)

XML file '/data/project/tae/gcam-core/input/policy/ndc/transportation/K_pass_ep.xml' created successfully with proper indentation and no extra newlines.
